# Medicinal Plant Recognition - EfficientNetB3
**Colab Training Notebook** — Run on GPU for best results (30-45 min)


In [ ]:
# ============================================================
# STEP 1: CHECK GPU (MUST show a GPU device!)
# ============================================================
# If this shows [] or "No GPU", go to:
# Runtime -> Change runtime type -> T4 GPU -> Save
# Then restart and run again.
# ============================================================

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU devices: {gpus}")

if not gpus:
    print("\n" + "!"*60)
    print("  *** NO GPU DETECTED! ***")
    print("  Training will be EXTREMELY SLOW (12+ hours).")
    print("  Go to: Runtime -> Change runtime type -> T4 GPU")
    print("!"*60)
else:
    print(f"\n[OK] GPU ready: {gpus[0].name}")
    print("   Training should take ~30-45 minutes.")


In [ ]:
# ============================================================
# STEP 2: IMPORTS
# ============================================================
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications.efficientnet import EfficientNetB3, preprocess_input

import warnings
warnings.filterwarnings('ignore')
print("[OK] All imports done.")


In [ ]:
# ============================================================
# STEP 3: MOUNT GOOGLE DRIVE & EXTRACT DATASET
# ============================================================
from google.colab import drive
import zipfile

drive.mount('/content/drive')

# === UPDATE THIS if your ZIP has a different name ===
DRIVE_ZIP_PATH = "/content/drive/MyDrive/IMFI Indian Medicinal Flower Image Dataset.zip"
EXTRACT_DIR = "/content/dataset"

# Create output directory on Drive (persists after disconnect!)
SAVE_DIR = "/content/drive/MyDrive/Mini Project Output"
os.makedirs(SAVE_DIR, exist_ok=True)

if os.path.exists(DRIVE_ZIP_PATH):
    if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) == 0:
        print(f"Extracting {DRIVE_ZIP_PATH}...")
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as z:
            z.extractall(EXTRACT_DIR)
        print("[OK] Dataset extracted!")
    else:
        print("[OK] Dataset already extracted.")
else:
    print(f"[ERROR] ZIP not found at: {DRIVE_ZIP_PATH}")
    print("   Update DRIVE_ZIP_PATH above to match your file name.")


In [ ]:
# ============================================================
# STEP 4: AUTO-DETECT DATASET ROOT
# ============================================================
def find_dataset_root(base_dir):
    for root, dirs, files_list in os.walk(base_dir):
        if len(dirs) >= 10:
            sample_dir = os.path.join(root, dirs[0])
            if os.path.isdir(sample_dir):
                sample_files = os.listdir(sample_dir)
                img_files = [f for f in sample_files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                if len(img_files) > 0:
                    return root
    return base_dir

DATA_DIR = find_dataset_root(EXTRACT_DIR)
print(f"[OK] Dataset root: {DATA_DIR}")
print(f"   Classes found: {len(os.listdir(DATA_DIR))}")


In [ ]:
# ============================================================
# STEP 5: LOAD DATASET
# ============================================================
def loading_the_data(data_dir):
    filepaths = []
    labels = []
    folds = sorted(os.listdir(data_dir))
    for fold in folds:
        foldpath = os.path.join(data_dir, fold)
        if not os.path.isdir(foldpath):
            continue
        filelist = os.listdir(foldpath)
        for file in filelist:
            fpath = os.path.join(foldpath, file)
            if os.path.isfile(fpath):
                filepaths.append(fpath)
                labels.append(fold)
    Fseries = pd.Series(filepaths, name='filepaths')
    Lseries = pd.Series(labels, name='labels')
    return pd.concat([Fseries, Lseries], axis=1)

df = loading_the_data(DATA_DIR)
print(f"[OK] Total images: {len(df)}")
print(f"   Number of classes: {df['labels'].nunique()}")
print(f"\nClasses: {sorted(df['labels'].unique())}")


In [ ]:
# ============================================================
# STEP 6: SPLIT DATA (80% train, 10% val, 10% test)
# ============================================================
train_df, temp_df = train_test_split(df, train_size=0.8, shuffle=True, random_state=42, stratify=df['labels'])
valid_df, test_df = train_test_split(temp_df, train_size=0.5, shuffle=True, random_state=42, stratify=temp_df['labels'])
print(f"[OK] Split: {len(train_df)} train / {len(valid_df)} validation / {len(test_df)} test")


In [ ]:
# ============================================================
# STEP 7: DATA GENERATORS (with preprocess_input — NOT rescale!)
# ============================================================
BATCH_SIZE = 32  # Larger batch for GPU speed
IMG_SIZE = (224, 224)

tr_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

ts_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = tr_gen.flow_from_dataframe(
    train_df, x_col='filepaths', y_col='labels',
    target_size=IMG_SIZE, class_mode='categorical',
    color_mode='rgb', shuffle=True, batch_size=BATCH_SIZE
)

valid_gen = ts_gen.flow_from_dataframe(
    valid_df, x_col='filepaths', y_col='labels',
    target_size=IMG_SIZE, class_mode='categorical',
    color_mode='rgb', shuffle=True, batch_size=BATCH_SIZE
)

test_gen = ts_gen.flow_from_dataframe(
    test_df, x_col='filepaths', y_col='labels',
    target_size=IMG_SIZE, class_mode='categorical',
    color_mode='rgb', shuffle=False, batch_size=BATCH_SIZE
)

# Save class indices
class_indices = train_gen.class_indices
class_indices_path = os.path.join(SAVE_DIR, 'medicinal_class_indices.json')
with open(class_indices_path, 'w') as f:
    json.dump(class_indices, f, indent=2)

num_classes = len(class_indices)
print(f"\n[OK] {num_classes} classes")
print(f"[OK] Class indices saved to: {class_indices_path}")


In [ ]:
# ============================================================
# STEP 8: BUILD MODEL
# ============================================================
def dense_block(dense, units, dropout_rate, act='relu'):
    x = Dense(units, activation=act)(dense)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    return x

img_shape = (224, 224, 3)
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=img_shape, pooling=None)
base_model.trainable = False  # Freeze for Phase 1

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = dense_block(x, 256, 0.5)
x = dense_block(x, 128, 0.3)
x = dense_block(x, 64, 0.2)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adamax(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

total_params = model.count_params()
trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
print(f"[OK] Model built!")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable: {trainable:,}")
print(f"   Non-trainable: {total_params - trainable:,}")


In [ ]:
# ============================================================
# STEP 9: PHASE 1 — Train classification head (base frozen)
# ============================================================
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'medicinal_model.keras')

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)
]

INITIAL_EPOCHS = 30

print(f"Phase 1: Training classification head ({INITIAL_EPOCHS} epochs)...")
print("Base model layers: FROZEN\n")

history1 = model.fit(
    train_gen,
    epochs=INITIAL_EPOCHS,
    validation_data=valid_gen,
    callbacks=callbacks,
    verbose=1
)
print("\n[OK] Phase 1 complete! Model saved to Google Drive.")


In [ ]:
# ============================================================
# STEP 10: PHASE 2 — Fine-tune top layers
# ============================================================
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=Adamax(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

FINE_TUNE_EPOCHS = 20

print(f"Phase 2: Fine-tuning top layers ({FINE_TUNE_EPOCHS} epochs)...\n")

history2 = model.fit(
    train_gen,
    epochs=FINE_TUNE_EPOCHS,
    validation_data=valid_gen,
    callbacks=callbacks,
    verbose=1
)
print("\n[OK] Phase 2 complete! Model saved to Google Drive.")


In [ ]:
# ============================================================
# STEP 11: EVALUATE
# ============================================================
print("Evaluating on test set...\n")
test_loss, test_accuracy = model.evaluate(test_gen, verbose=1)

print(f"\n{'='*50}")
print(f"  TEST ACCURACY: {test_accuracy:.2%}")
print(f"  TEST LOSS:     {test_loss:.4f}")
print(f"{'='*50}")


In [ ]:
# ============================================================
# STEP 12: SAVE FINAL MODEL TO GOOGLE DRIVE
# ============================================================
model.save(MODEL_SAVE_PATH)

# Also save class indices
with open(os.path.join(SAVE_DIR, 'medicinal_class_indices.json'), 'w') as f:
    json.dump(class_indices, f, indent=2)

print(f"\n[OK] Model saved to: {MODEL_SAVE_PATH}")
print(f"[OK] Class indices saved to: {SAVE_DIR}/medicinal_class_indices.json")
print(f"\n[INFO] Check your Google Drive -> 'Mini Project Output' folder")
print(f"   Download both files to your PC and run: python test_model.py")


In [ ]:
# ============================================================
# STEP 13: QUICK TEST (optional)
# ============================================================
from tensorflow.keras.preprocessing import image as keras_image

labels = {v: k for k, v in class_indices.items()}

# Test on a random image from the test set
sample = test_df.sample(5, random_state=42)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for idx, (_, row) in enumerate(sample.iterrows()):
    img = keras_image.load_img(row['filepaths'], target_size=(224, 224))
    img_array = keras_image.img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array, verbose=0)
    pred_idx = np.argmax(preds[0])
    pred_name = labels[pred_idx]
    confidence = preds[0][pred_idx]
    true_name = row['labels']
    color = 'green' if pred_name == true_name else 'red'

    axes[idx].imshow(keras_image.load_img(row['filepaths']))
    axes[idx].set_title(f"Pred: {pred_name}\n({confidence:.1%})\nTrue: {true_name}", fontsize=8, color=color)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'test_predictions.png'), dpi=100)
plt.show()
print("\n[OK] Test predictions saved to Drive!")


In [ ]:
# ============================================================
# STEP 14: GENERATE COMPREHENSIVE METRICS REPORTS (PNG & TXT)
# ============================================================
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("Generating comprehensive metrics...\n")
test_gen.reset()
preds = model.predict(test_gen, verbose=1)
y_pred = np.argmax(preds, axis=1)
y_true = test_gen.classes

labels_map = {v: k for k, v in class_indices.items()}
class_names = [labels_map[i] for i in range(num_classes)]

report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

# 1. Plot F1, Precision, Recall Bar Chart
classes = [c for c in class_names if c in report_dict]
precision = [report_dict[c]['precision'] for c in classes]
recall = [report_dict[c]['recall'] for c in classes]
f1 = [report_dict[c]['f1-score'] for c in classes]

x = np.arange(len(classes))
width = 0.25
fig, ax = plt.subplots(figsize=(16, 8))
ax.bar(x - width, precision, width, label='Precision', color='#4C72B0')
ax.bar(x, recall, width, label='Recall', color='#55A868')
ax.bar(x + width, f1, width, label='F1-Score', color='#C44E52')

ax.set_ylabel('Scores', fontsize=12)
ax.set_title('Precision, Recall, and F1-Score by Class', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=10)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.set_ylim([0, 1.1])
plt.tight_layout()
metrics_path = os.path.join(SAVE_DIR, 'class_metrics_bar_chart.png')
plt.savefig(metrics_path, dpi=200)
plt.show()

# 2. Plot Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(18, 16))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, annot_kws={'size': 8})
plt.title('Confusion Matrix', fontsize=20, fontweight='bold', pad=20)
plt.ylabel('True Class', fontsize=14)
plt.xlabel('Predicted Class', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
cm_path = os.path.join(SAVE_DIR, 'confusion_matrix_heatmap.png')
plt.savefig(cm_path, dpi=200)
plt.show()

# 3. Save Text Report
report_text = classification_report(y_true, y_pred, target_names=class_names)
txt_path = os.path.join(SAVE_DIR, 'full_classification_report.txt')
with open(txt_path, 'w') as f:
    f.write("="*60 + "\nMODEL EVALUATION METRICS REPORT\n" + "="*60 + "\n\n")
    f.write(report_text)
    f.write(f"\n\nAccuracy: {report_dict['accuracy']:.4f}\n")
    f.write(f"Macro Avg F1: {report_dict['macro avg']['f1-score']:.4f}\n")
    f.write(f"Weighted Avg F1: {report_dict['weighted avg']['f1-score']:.4f}\n")

print(f"\n[OK] Metric reports saved to Google Drive: {SAVE_DIR}")
